# 05 — Arquiteturas para Documentos Longos + Comparativo Final

Duas abordagens incrementais para lidar com a limitação de 512 tokens do BERT
em votos longos do TCU (2.000–10.000 tokens):

1. **Head+Tail** — concatena os primeiros 256 tokens + últimos 254 tokens do voto.
   Hipótese: o início contextualiza (relator, objeto) e o final concentra a análise
   conclusiva (antes do dispositivo já removido por `construir_feature_voto`).

2. **Hierárquico** — encoder BERT por sentença + attention ponderada sobre sentenças.
   Processa o documento inteiro (até 32 sentenças × 128 tokens = 4.096 tokens)
   com custo linear no número de sentenças. Encoder chamado em chunks de 16
   sentenças para caber em GPU T4 (15 GB).

Ambos usam **pesos de classe** (cross-entropy ponderada) e o mesmo LegalBert-pt.

**Requer GPU** (T4 suficiente). Tempo estimado: ~2h (head+tail) + ~3h (hierárquico).

**Saída:** `resultados/metricas_hierarquico.json` + tabela comparativa final com
todos os modelos (02, 03, 04, 05).

## 1. Setup

In [ ]:
import os, sys, subprocess
REPO_DIR = os.environ.get('REPO_DIR', '/content/deep-acordao-tcu2')
REPO_URL = 'https://github.com/bsousa7/deep-acordao-tcu2.git'
BRANCH = os.environ.get('BRANCH', 'claude/deep-acordao-tcu-refactor-yyjfr3')
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    subprocess.run(['git', 'clone', REPO_URL, '--branch', BRANCH, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('repo em', REPO_DIR)

In [ ]:
%pip -q install "transformers>=4.44" "peft>=0.13" "datasets>=2.20" accelerate scikit-learn pyarrow scipy nltk
import torch
print('deps ok | torch', torch.__version__)

## 2. Remove torchao incompatível

In [ ]:
import importlib, subprocess

if importlib.util.find_spec('torchao') is not None:
    subprocess.run(['pip', 'uninstall', '-y', 'torchao'], check=True)
    print('torchao removido — OK')
else:
    print('torchao já ausente — OK')

## 3. GPU + Configuração

In [ ]:
import torch
USAR_GPU = torch.cuda.is_available()
if USAR_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: GPU não detectada — este notebook requer GPU para resultados finais.')
    print('Ative em Runtime > Alterar tipo de ambiente > T4.')

In [ ]:
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

RANDOM_STATE = 42

# --- Config adaptativa ---
if USAR_GPU:
    N_SPLITS = 5
    # Head+Tail
    HT_HEAD = 256
    HT_TAIL = 254
    HT_EPOCHS = 5
    HT_BATCH = 16
    # Hierárquico (reduzido para caber em T4 15GB)
    HIER_MAX_SENTS = 32
    HIER_SENT_LEN = 128
    HIER_EPOCHS = 5
    HIER_BATCH = 2
    print('GPU detectada -> config completa')
else:
    N_SPLITS = 3
    HT_HEAD = 64
    HT_TAIL = 62
    HT_EPOCHS = 1
    HT_BATCH = 8
    HIER_MAX_SENTS = 8
    HIER_SENT_LEN = 64
    HIER_EPOCHS = 1
    HIER_BATCH = 2
    print('SEM GPU -> config reduzida (validação de fluxo apenas)')

BASE = Path(REPO_DIR)

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / 'deep-acordao-tcu2'
except Exception as _e:
    DRIVE_ROOT = None
    print(f'Google Drive indisponível: {_e}')

_drive_interim = (DRIVE_ROOT / 'data' / 'interim') if DRIVE_ROOT else None
PERSIST_BASE = DRIVE_ROOT if (_drive_interim and _drive_interim.exists()) else BASE

DATA_INTERIM = PERSIST_BASE / 'data' / 'interim'
RESULTADOS = PERSIST_BASE / 'resultados'
FIGURAS = RESULTADOS / 'figuras'; FIGURAS.mkdir(parents=True, exist_ok=True)

print(f'Lendo/gravando dados em: {PERSIST_BASE}')

## 4. Carrega corpus

In [ ]:
import pandas as pd
import numpy as np

def _checar_arquivo(caminho, persist_base):
    if not caminho.exists():
        raise FileNotFoundError(
            f"Não encontrei {caminho}.\n"
            f"PERSIST_BASE resolvido para: {persist_base}\n"
            "Rode o notebook 01 primeiro (ou monte o Google Drive)."
        )

_arq = DATA_INTERIM / 'acordaos_rotulados.parquet'
_checar_arquivo(_arq, PERSIST_BASE)
df = pd.read_parquet(_arq)
print('corpus n =', len(df))
print(df['LABEL'].value_counts().to_string())

# Diagnóstico: comprimento dos votos em tokens (aproximação por espaços)
lens = df['VOTO_LIMPO'].fillna('').str.split().str.len()
print(f'\nComprimento dos votos (palavras):')
print(f'  mediana = {lens.median():.0f} | média = {lens.mean():.0f} | p95 = {lens.quantile(0.95):.0f}')
print(f'  > 512 palavras: {(lens > 512).sum()} ({100*(lens>512).mean():.1f}%)')
print(f'  > 1024 palavras: {(lens > 1024).sum()} ({100*(lens>1024).mean():.1f}%)')

## 5. Estratégia 1 — Head+Tail (LegalBert + LoRA)

Tokeniza cada voto como `[CLS] primeiros_256_tokens [SEP] últimos_254_tokens [SEP]`.
Total = 512 tokens, mas capturando informação do final do voto que a truncagem
simples perde. Hipótese: a análise conclusiva (últimos parágrafos antes do
dispositivo removido) contém sinais discriminativos fortes.

In [ ]:
from src.modelos.hierarquico import kfold_head_tail

print(f'Head+Tail: head={HT_HEAD}, tail={HT_TAIL}, epochs={HT_EPOCHS}, '
      f'batch={HT_BATCH}, n_splits={N_SPLITS}')
print('Iniciando K-Fold...')

res_ht = kfold_head_tail(
    df, campo='VOTO_LIMPO', n_splits=N_SPLITS,
    head_tokens=HT_HEAD, tail_tokens=HT_TAIL,
    epochs=HT_EPOCHS, batch_size=HT_BATCH,
    usar_lora=True,
)
print(f"\n{'='*50}")
print(f"HEAD+TAIL — F1-macro = {res_ht['mean_f1']:.4f}  IC95={res_ht['ci_95']}")
print(f"Acurácia média = {res_ht['mean_acc']:.4f}")
print(f"Fold scores: {[f'{s:.4f}' for s in res_ht['fold_scores']]}")

## 6. Estratégia 2 — Modelo Hierárquico

Arquitetura em dois níveis:
- **Nível 1 (sentença):** encoder BERT gera representação CLS por sentença.
- **Nível 2 (documento):** attention ponderada sobre as representações de sentenças.

Vantagens: processa o voto inteiro (~6K tokens), attention interpretável por
sentença, custo linear no número de sentenças.

Estratégia de treino:
- Época 1: encoder congelado (warm-up do attention + classifier).
- Épocas 2+: fine-tune completo com lr diferenciado (encoder 10× menor).

In [ ]:
from src.modelos.hierarquico import kfold_hierarquico

print(f'Hierárquico: max_sents={HIER_MAX_SENTS}, sent_len={HIER_SENT_LEN}, '
      f'epochs={HIER_EPOCHS}, batch={HIER_BATCH}, n_splits={N_SPLITS}')
print('Iniciando K-Fold...')

res_hier = kfold_hierarquico(
    df, campo='VOTO_LIMPO', n_splits=N_SPLITS,
    max_sents=HIER_MAX_SENTS, max_sent_len=HIER_SENT_LEN,
    epochs=HIER_EPOCHS, batch_size=HIER_BATCH,
)
print(f"\n{'='*50}")
print(f"HIERÁRQUICO — F1-macro = {res_hier['mean_f1']:.4f}  IC95={res_hier['ci_95']}")
print(f"Acurácia média = {res_hier['mean_acc']:.4f}")
print(f"Fold scores: {[f'{s:.4f}' for s in res_hier['fold_scores']]}")

## 7. Comparativo Final — Todos os Modelos

Consolida resultados de todos os notebooks (02–05) numa tabela única,
ordenada por F1-macro.

In [ ]:
import json
from src.avaliacao.metricas import comparar_modelos

todos = {}

# Resultados deste notebook (só inclui se já rodou)
if 'res_ht' in dir():
    todos['LegalBert Head+Tail (05)'] = res_ht
if 'res_hier' in dir():
    todos['LegalBert Hierárquico (05)'] = res_hier

# Carrega resultados dos notebooks anteriores
caminho_baseline = RESULTADOS / 'metricas_baseline.json'
if caminho_baseline.exists():
    m = json.loads(caminho_baseline.read_text())
    todos['TF-IDF + LogReg (02)'] = m['kfold_5x_balanced']
else:
    print(f'Aviso: {caminho_baseline} não encontrado — rode notebook 02.')

caminho_textcnn = RESULTADOS / 'metricas_textcnn.json'
if caminho_textcnn.exists():
    m = json.loads(caminho_textcnn.read_text())
    todos['TextCNN (03)'] = m['kfold_5x']
else:
    print(f'Aviso: {caminho_textcnn} não encontrado — rode notebook 03.')

caminho_legalbert = RESULTADOS / 'metricas_legalbert.json'
if caminho_legalbert.exists():
    m = json.loads(caminho_legalbert.read_text())
    todos['LegalBert Truncado (04)'] = m['kfold']
else:
    print(f'Aviso: {caminho_legalbert} não encontrado — rode notebook 04.')

if not todos:
    raise RuntimeError('Nenhum resultado disponível para comparar.')

tabela = comparar_modelos(todos)
print('\n' + '='*70)
print('COMPARATIVO FINAL — F1-macro (K-Fold, VOTO_LIMPO, pesos de classe)')
print('='*70)
print(tabela.to_string(index=False))
print('\nNota: todos os modelos usam exclusivamente VOTO_LIMPO (sem leakage).')
print('Pesos de classe aplicados em todos os treinos.')

## 8. Visualização comparativa

In [ ]:
import matplotlib.pyplot as plt

modelos = tabela['Modelo/Campo'].tolist()
f1s = tabela['F1-macro'].tolist()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(modelos)), f1s, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336'][:len(modelos)])
ax.set_yticks(range(len(modelos)))
ax.set_yticklabels(modelos, fontsize=10)
ax.set_xlabel('F1-macro')
ax.set_title('Comparativo de Modelos — VOTO_LIMPO (sem leakage, com pesos de classe)')
ax.set_xlim(0, max(f1s) * 1.15)

for bar, val in zip(bars, f1s):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.axvline(x=0.333, color='gray', linestyle='--', alpha=0.5, label='Random (3 classes)')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(FIGURAS / 'comparativo_final.png', dpi=150)
plt.show()
print(f'Figura salva: {FIGURAS / "comparativo_final.png"}')

## 9. Análise: impacto do comprimento do documento

Compara o desempenho do head+tail vs truncagem padrão em função do
comprimento do voto — documentos longos devem se beneficiar mais.

In [ ]:
# Análise descritiva: quanto do voto é perdido com truncagem a 512 tokens?
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('dominguesm/legal-bert-base-cased-ptbr')
sample = df['VOTO_LIMPO'].fillna('').sample(min(500, len(df)), random_state=42)
tok_lens = sample.apply(lambda t: len(tokenizer.encode(t, add_special_tokens=False)))

print('Comprimento em tokens (amostra de 500 votos):')
print(f'  mediana = {tok_lens.median():.0f}')
print(f'  média = {tok_lens.mean():.0f}')
print(f'  p75 = {tok_lens.quantile(0.75):.0f}')
print(f'  p95 = {tok_lens.quantile(0.95):.0f}')
print(f'  max = {tok_lens.max()}')
print(f'\n  Truncados (>510 tokens): {(tok_lens > 510).sum()} ({100*(tok_lens>510).mean():.1f}%)')
print(f'  Perda média nos truncados: {(tok_lens[tok_lens>510] - 510).mean():.0f} tokens perdidos')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tok_lens.clip(upper=5000), bins=50, color='steelblue', edgecolor='white')
ax.axvline(510, color='red', linestyle='--', label='Limite BERT (512)')
ax.axvline(510, color='red', linestyle='--')
ax.set_xlabel('Número de tokens')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de comprimento dos votos (tokens BERT)')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURAS / 'distribuicao_tokens.png', dpi=150)
plt.show()

## 10. Persistência

In [ ]:
from src.avaliacao.metricas import salvar_json

saida = {
    'modelo': 'Arquiteturas para Documentos Longos',
    'feature': 'VOTO_LIMPO',
    'usar_gpu': USAR_GPU,
}

if 'res_ht' in dir():
    saida['head_tail'] = {
        'estrategia': 'head+tail',
        'hiperparametros': {
            'head_tokens': HT_HEAD, 'tail_tokens': HT_TAIL,
            'epochs': HT_EPOCHS, 'batch_size': HT_BATCH,
            'n_splits': N_SPLITS, 'usar_lora': True,
        },
        'mean_f1': res_ht['mean_f1'],
        'std_f1': res_ht['std_f1'],
        'ci_95': list(res_ht['ci_95']),
        'mean_acc': res_ht['mean_acc'],
        'fold_scores': res_ht['fold_scores'],
    }

if 'res_hier' in dir():
    saida['hierarquico'] = {
        'estrategia': 'encoder por sentença + attention',
        'hiperparametros': {
            'max_sents': HIER_MAX_SENTS, 'max_sent_len': HIER_SENT_LEN,
            'epochs': HIER_EPOCHS, 'batch_size': HIER_BATCH,
            'n_splits': N_SPLITS, 'freeze_encoder_epochs': 1,
        },
        'mean_f1': res_hier['mean_f1'],
        'std_f1': res_hier['std_f1'],
        'ci_95': list(res_hier['ci_95']),
        'mean_acc': res_hier['mean_acc'],
        'fold_scores': res_hier['fold_scores'],
    }

salvar_json(saida, RESULTADOS / 'metricas_hierarquico.json')
tabela.to_csv(RESULTADOS / 'comparacao_final.csv', index=False)

print(f'OK — persistido em: {PERSIST_BASE}')
print(f'  metricas_hierarquico.json')
print(f'  comparacao_final.csv')
if not USAR_GPU:
    print('\nLEMBRETE: execução sem GPU — números são apenas validação de fluxo.')

## 11. Discussão

### Interpretação dos resultados

| Modelo | Cobertura do voto | Hipótese |
|--------|-------------------|----------|
| LegalBert Truncado (04) | ~400 palavras iniciais | Início do voto é suficiente |
| LegalBert Head+Tail (05) | ~250 iniciais + ~250 finais | Final contém sinais discriminativos |
| Hierárquico (05) | ~6.000 tokens (48 sentenças) | Todo o voto contribui (desigualmente) |
| TF-IDF + LogReg (02) | Voto completo (bag-of-words) | Vocabulário discriminativo basta |

### Por que o baseline pode superar deep learning neste corpus?

1. **Corpus pequeno** (~3.600 docs) com classes extremamente raras (~100 exemplos
   para Regular) — insuficiente para fine-tuning robusto de 110M parâmetros.
2. **TF-IDF captura o voto inteiro** sem truncagem, e LogReg com regularização
   L2 generaliza bem em alta dimensionalidade.
3. **O sinal discriminativo pode ser lexical** (presença/ausência de termos
   específicos) mais do que semântico — favorece bag-of-words.

### Contribuição para a dissertação

A comparação demonstra que:
- Eliminar o vazamento (SUMARIO → VOTO_LIMPO) torna a tarefa genuinamente difícil.
- O desbalanceamento extremo (91/6.6/2.6%) é o gargalo principal.
- Modelos mais complexos não necessariamente superam baselines simples em
  corpora pequenos e desbalanceados — resultado válido e publicável.